# ENVRI HUB library usage
In this notebook we'll showcase the intended usage of the *VRE-Lib* component developed by WP 13 and WP 14.
The VRE-Lib is available as a *Python package* on the [public package index](https://pypi.org/project/envrihub/) un the *envrihub* name.
This means it can be installed with:

In [ ]:
! pip install --upgrade envrihub==0.1.3
! pip install openapi-spec-validator

The main element you have to import is the _Hub_ object:

In [ ]:
from envrihub import Hub

hub = Hub()

## Free text search
The _Hub_ object offers several search and filter features, exactly like in the Catalogue of Services graphical interface. You can query it with a free text search (that checks all the metadata descriptions in the CoS):

In [ ]:
for res in hub.search_catalogue('sealevel'):
    print(res.title)
    print(f'\t resource id: {res.uid}')
    print(f'\t resource type: {res.type}')
    print(f'\t resurce description: {res.description}\n\n')

The `search_catalogue` method always return a *Generator* object, i.e. somthing you can iterate on, so you can process search results one by one as they get fetched from the catalogue.

Each result has the shape of a `Distribution` object, containing anything you need to work with that resource, including the data access logic. We'll see that in the detail later, for now it's enough to know that all of them have an `id` that is unique in the Catalogue and there are two types of them: *Web Services*, and *Downloadable Files*.
## Spatial coverage search
We can query the catalogue also by specifying a region of interest. Such a region can be expressed as a WKT string, a format used by most GIS applications, so you can copy-paste spatial information from your GIS of choice and shove it straight into the `search_catalogue` function.

In [ ]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

geography = 'POLYGON((10.70 48.34,18.98 48.34, 18.98 42.17, 10.70 42.17, 10.70 48.34))'

for res in hub.search_catalogue(geography = geography):
    print(res.title)
    print(f'\t resource id: {res.uid}')
    print(f'\t resource type: {res.type}')
    print(f'\t resurce description: {res.description}\n\n')
    

## EXV and temporal coverage
We can think of other kinds of facets, like time boundaries, or specific variables of intersest.
The `exv` field accepts as value any of the [known EXVs](https://catalogue.staging.envri.eu/api/v1/resources/exvs) in the ENVRI-Hub catalogue 

In [ ]:
for res in hub.search_catalogue(exv = "https://vocab.nerc.ac.uk/collection/EXV/current/EXV028/",
                               start_date = '2024-01-01', end_date = '2025-01-01'):
    print(res.title)
    print(f'\t resource id: {res.uid}')
    print(f'\t resource type: {res.type}')
    print(f'\t resurce description: {res.description}\n\n')

# Accessing data
Now let's talk about *data access*. Remember the `Distribution` object? It contains all you need to access data and to work with it, and you can get one either by *searching* the catalogue as we saw above, or by retrieving one directly using its `uid`, like in the following example.

In [ ]:
service_id = 'file:///ArgoBGC_distribution_005'

res = hub.fetch_from_catalogue(service_id)
print(res.title)

Where do you get IDs? Either browsing the catlogue on your own, or with the `search_catalogue` method.

What we have now inside the `res` variable is a `Distribution` object that has the following attributes:
+ `title`: the resource title/display name
+ `uid`: the unique internal identifier to fetch it a later time
+ `description`: a human readable minimal description
+ `type`: whether it is a web service or a file
+ `href`: a link to more metadata
+ `service_documentation`: a link to some material that should allow you to get a hold on how to use the resouce.
+ `metadata`: all the metadata avaiable.

Plus the `is_downloadable` function that answers the fundamental question for all you digital kleptomaniacs: *can I download it on my laptop?*

In [ ]:
res.metadata

Quite some information isn't it?
That's becasue you might want to know what you'll find inside the data *before* opening it. You know... To avoid jumpscares.

Now it's finally time to access the *actual data* with the `dao` attribute, that *always* comes with helpful documentation you can access with the *help()* function or with third party Jupyter extensions.

In [ ]:
catalogue_dao = res.dao
help(catalogue_dao)

And now let's do some magic with data. The [Environmental Expert](https://chat.envri.eu/) can help you with this.

In [ ]:
import pandas as pd
import json

service_response = catalogue_dao.access(latitude_less='33', latitude_more='29', longitude_less='30', longitude_more='-21', pres_less='10', time_less='2025-02-01T00:00:00Z', pres_more='0', time_more='2025-01-01T00:00:00Z')

# 1. The service returns a geoJSON, let's load it
data = json.loads(service_response)

# 2. Extract features
features = data['features']

# 3. Transform 'properties' into dictionaries

rows = []
for f in features:
    # Get all properties (temp, pres, psal, ecc.)
    record = f['properties'].copy()
    
    # Extract coordinates: [longitudine, latitudine]
    record['longitude'] = f['geometry']['coordinates'][0]
    record['latitude'] = f['geometry']['coordinates'][1]
    
    rows.append(record)

# 4. Create DataFrame
df = pd.DataFrame(rows)

# 5. Remove empty column
df = df.replace("", float('nan'))
df_pulito = df.dropna(axis=1, how='all')

# Print result
print(f"Clean table: {df_pulito.shape[0]} rows and {df_pulito.shape[1]} columns.")
print(df_pulito.head())

More magic: let's draw a graph

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load data
#with open('dati_servizio.bin', 'r') as f:
data = json.loads(service_response)

# Extract properties
features = data['features']
df = pd.DataFrame([f['properties'] for f in features])

# Conversion and clean up
df['pres'] = pd.to_numeric(df['pres'], errors='coerce')
df['doxy'] = pd.to_numeric(df['doxy'], errors='coerce')
df_plot = df.dropna(subset=['pres', 'doxy'])

# Create Scatter Chart
plt.scatter(df_plot['doxy'], df_plot['pres'], alpha=0.5)
plt.xlabel('Dissolved Oxygen (doxy)')
plt.ylabel('Pressure (pres)')
plt.title('Scatter Chart: Pressure vs Dissolved Oxygen')
plt.grid(True, linestyle='--', alpha=0.7)
plt.savefig('scatter_pres_doxy.png')